In [0]:
# workspace.silver.crm_customers
# workspace.silver.crm_product
# workspace.silver.crm_sales_details
# workspace.silver.erp_customer
# workspace.silver.erp_location
# workspace.silver.erp_product_category

#### # 1. Multi-Table Ingestion & Clean Silver Setup

In [0]:
import pyspark.sql.functions as F

# Ingest your clean, refactored Silver layer product tables using explicit alias qualifiers
crm_prod = spark.table("workspace.silver.crm_product").alias("pn")
erp_cat  = spark.table("workspace.silver.erp_product_category").alias("pc")

#### # 2. Filter Active Records, Relational Joins & Windowing Logic

In [0]:
from pyspark.sql.window import Window

# 1. Filter out historic changes and execute Left Join on verified catalog id (pc.id)
joined_df = (
    crm_prod
    .filter(F.col("pn.end_date").isNull())
    .join(erp_cat, F.col("pn.category_id") == F.col("pc.id"), "left")
)

# 2. Build your sequential surrogate key tracking over start_date and product_number
# OPTION A (Standard): Generates sequential IDs (keeps the warning)
transformed_df = joined_df.withColumn(
    "product_key", 
    F.row_number().over(Window.orderBy("pn.start_date", "pn.product_number"))
)

# OPTION B (Optimized): Uncomment the line below to eliminate the warning completely:
# transformed_df = joined_df.withColumn("product_key", F.monotonically_increasing_id())

#### # 3. Dimensional Schema Ordering, Casting, and Gold Table Storage

In [0]:
# Single-Pass Operation: Extracts specific alias attributes, enforces type-casting, and outputs final names
final_df = transformed_df.select(
    F.col("product_key").cast("integer").alias("product_key"),
    F.col("pn.product_id").cast("string").alias("product_id"),
    F.col("pn.product_number").cast("string").alias("product_number"),
    F.col("pn.product_name").cast("string").alias("product_name"),
    F.col("pn.category_id").cast("string").alias("category_id"),
    F.col("pc.category").cast("string").alias("category"),
    F.col("pc.subcategory").cast("string").alias("subcategory"),
    F.col("pc.maintenance").cast("string").alias("maintenance"),
    F.col("pn.product_cost").cast("double").alias("cost"), # Fixed to product_cost
    F.col("pn.product_line").cast("string").alias("product_line"),
    F.col("pn.start_date").cast("date").alias("start_date")
)

# Commit the finalized schema directly as a production-grade Gold Delta table
final_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_products")

# Display a clean data preview to inspect column alignment and verify rows
final_df.display()